# Stage evaluation: how good is every section of the pipeline?

`Single_Image_Run.ipynb` answers "what score do we get and how long does it take". This notebook asks the question one level deeper: **which section loses the points, and how much?** It uses the same pipeline pieces, one photo at a time, but looks at what each stage hands to the next.

**"Found" means: a place within 4 sigma of the truth** (the +3 point band). The strict version, within 1 sigma (+10 points), is always shown next to it. Both numbers are in `FOUND_SIGMA` and `STRICT_SIGMA`.

| Stage | Question | What is measured |
|---|---|---|
| 1. Image processing | Is the photo usable, and does it look like the map where it truly is? | valid share and contrast of the north-up canvas; edge strength; the score of every map at the true place against the rest of the search disc |
| 2. Coarse search | Is the target among the chosen peaks? | recall at K (target anywhere in the first K peaks), rank of the first right peak, does the global maximum of the score map lie at the target |
| 3. Semantic layer (DINO) | Does it help the first place, and does it keep the target when only M of K go on? | top-1 before and after (paired, with interval), recall at M of K, rank of the first right peak, how well the DINO score separates right from wrong places |
| 4. Fine matching | If the target is among the M, does the matcher find it? | inliers of the right candidate against the wrong ones, how often the right one has the most inliers, how often it has none, error after the shift correction |
| 5. Decision | Does it answer the right photos? | precision, answerable photos that were skipped, points lost to skipping and to wrong answers, bar sweep |
| 6. End to end | Where do the points go? | a funnel: all photos, target in K, target in M, right candidate chosen, answered, within 4 sigma, within 1 sigma |

**How it runs, in three passes.** Pass A (Part 3) renders each photo, computes the score maps, the K_MAX best peaks and the DINO scores once. Pass B (Part 4) runs the fine matcher on all K_MAX peaks of every photo once per matcher. Everything after that (the tables and the hyperparameter sweeps of K, M, DINO weight, answer bar, gamma, rescue rule) is calculated from the stored numbers, **without matching again**. Sweeping a hyperparameter therefore costs seconds.

**Read the sweeps with care.** They search many settings on the same 100 photos they are scored on, so the best row is optimistic. Trust a setting only if it also helps in both halves of the photos, and look for a flat region, not a single best row.

The analysis of *every* hyperparameter over wide ranges (borders, margins, confidence, RANSAC, strategies) is a separate notebook: `tests/Stage_Evaluation/Parameter_Test.ipynb`.


## Colab version

This is `Stage_Evaluation.ipynb` with two cells added and nothing else changed:

* **the first code cell** (run it first): mounts Drive, copies what the run needs from the repo copy on Drive to the local disk, builds the Python 3.10 environment, and starts a 3.10 kernel that every later code cell runs in;
* **the last cell**: copies the result tables to a separate folder on Drive.

Drive is only **read** from the repo copy. Nothing is written into it (if that folder syncs with your computer, your offline files stay as they are); the results go to `DRIVE_RESULTS`.

Why the copy takes long on Colab, and what this cell does about it: the two map scenes are 11.7 GB each as unpacked `.tif` files but only 3.6 and 3.3 GB as `.zip` files, so the zips are copied and unpacked on the local disk. `d1` is 501 small files, which are slow one by one on Drive, so they are copied 16 at a time. The weights are copied only for the matchers in `COLAB_MATCHERS`. The cell prints how long every copy took.

In [ ]:
#@title Colab: copy the data from Drive, build the Python 3.10 environment, start a 3.10 kernel (run this cell first)
# Steps, in order (the first time about 30 minutes; a later run in the same Colab session skips what already exists):
#   1. mount Google Drive and find the repo copy on it (Drive is only READ from; nothing is written into the repo copy)
#   2. copy to the local disk /content: the code, the weights of the matchers in COLAB_MATCHERS, d1, and the two map zips (unpacked locally)
#   3. build a Python 3.10 environment with the versions of requirements.txt (torch 2.8.0 + CUDA 12.6)
#   4. start one Python 3.10 kernel; every later code cell runs in it (a cell whose first line is  #main  runs in the normal Colab Python)
import os, sys, re, json, time, base64, queue, subprocess, glob, zipfile
from concurrent.futures import ThreadPoolExecutor

# ---------------- settings of this cell ----------------
DRIVE_REPO = ""                                             # the folder on Drive that holds the repo copy (it contains single_photo_pipeline.py). "" = search for it
DRIVE_RESULTS = "/content/drive/MyDrive/InterIIT_Colab_results"   # the result tables go here, never into the repo copy
LOCAL_REPO = "/content/geoloc"                              # where the working copy lives (fast local disk)
COLAB_MATCHERS = ["loftr_minima"]                           # must be the same list as MATCHERS in Part 1: only the weights of these matchers are copied
COPY_WORKERS = 4                                            # files copied at the same time (Drive is faster with several streams)
SCENES = ["268234931", "268234921"]                         # the two scenes of the new base map

assert len(COLAB_MATCHERS) >= 1 and all(m in ("lightglue", "loftr_minima", "roma_minima") for m in COLAB_MATCHERS), "COLAB_MATCHERS: choose from lightglue, loftr_minima, roma_minima"
assert COPY_WORKERS >= 1, "COPY_WORKERS must be 1 or more"
assert LOCAL_REPO.startswith("/content/"), "LOCAL_REPO must be on the local disk (/content/...)"


def sh(command):
    # runs a shell command, shows its output only when it fails
    result = subprocess.run(command, shell=True, capture_output=True, text=True, executable="/bin/bash")
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("command failed: " + command[:160])
    return result.stdout.strip()


# ---------------- 1. Google Drive ----------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive                          # exists only in the normal Colab kernel
    drive.mount("/content/drive")                           # a permission window opens: click through it once
assert os.path.isdir("/content/drive/MyDrive"), "Drive is not mounted"

if DRIVE_REPO == "":
    found = []
    for depth in range(1, 5):                               # look one folder level deeper each time, stop at the first hit
        found += glob.glob("/content/drive/MyDrive/" + "*/" * depth + "single_photo_pipeline.py")
        if len(found) > 0:
            break
    assert len(found) > 0, "single_photo_pipeline.py not found on Drive: set DRIVE_REPO to the repo copy folder"
    if len(found) > 1:
        print("several repo copies found, using the first; set DRIVE_REPO to choose:", found)
    DRIVE_REPO = os.path.dirname(found[0])
assert os.path.exists(DRIVE_REPO + "/single_photo_pipeline.py"), "DRIVE_REPO has no single_photo_pipeline.py: " + DRIVE_REPO
print("repo copy on Drive:", DRIVE_REPO)

# ---------------- 2. copy to the local disk ----------------
start = time.time()
sh('mkdir -p "%s" && rsync -rt --exclude=/dataset --exclude=/.git --exclude=__pycache__ --exclude="/venv*" "%s/" "%s/"' % (LOCAL_REPO, DRIVE_REPO, LOCAL_REPO))
print("code copied in %.0f s" % (time.time() - start))
for needed in ["single_photo_pipeline.py", "src", "train", "tests/Test_Iteration_1/results/overnight/state.json", "tests/Test_Iteration_1/results/overnight/models/f4.pt"]:
    assert os.path.exists(LOCAL_REPO + "/" + needed), "missing in the repo copy on Drive: " + needed + " (the fusion models are not in git: copy tests/Test_Iteration_1/results/overnight/ to the Drive copy)"


def copy_one(source, target):
    # copies one big file from Drive to the local disk (skipped when it is already there); returns (gigabytes, seconds)
    assert os.path.exists(source), "not on Drive: " + source
    if os.path.exists(target) and os.path.getsize(target) == os.path.getsize(source):
        return 0.0, 0.0
    os.makedirs(os.path.dirname(target), exist_ok=True)
    begin = time.time()
    sh('cp "%s" "%s.part" && mv "%s.part" "%s"' % (source, target, target, target))
    return os.path.getsize(target) / 1e9, time.time() - begin


def get_scene(scene):
    # one map scene: the zip is 3.5 times smaller than the tif, so copy the zip and unpack the tif on the local disk
    folder = LOCAL_REPO + "/dataset/Dataset_Official_2"
    tif = folder + "/" + scene + "/BAND.tif"
    if os.path.exists(tif):
        return scene, 0.0, 0.0, 0.0
    zip_drive = DRIVE_REPO + "/dataset/Dataset_Official_2/" + scene + ".zip"
    if not os.path.exists(zip_drive):
        gigabytes, seconds = copy_one(DRIVE_REPO + "/dataset/Dataset_Official_2/" + scene + "/BAND.tif", tif)
        return scene, gigabytes, seconds, 0.0
    zip_local = "/content/zips/" + scene + ".zip"
    gigabytes, seconds = copy_one(zip_drive, zip_local)
    begin = time.time()
    with zipfile.ZipFile(zip_local) as archive:
        archive.extract(scene + "/BAND.tif", "/content/unzip_tmp")           # unpacked next to the final place, then moved: no half-written tif
    os.makedirs(os.path.dirname(tif), exist_ok=True)
    os.replace("/content/unzip_tmp/" + scene + "/BAND.tif", tif)
    os.remove(zip_local)
    return scene, gigabytes, seconds, time.time() - begin


def get_d1():
    # the 501 photos: many small files, copied 16 at a time
    target = LOCAL_REPO + "/dataset/d1"
    if os.path.exists(target + "/images/metadata.csv"):
        return 0.0, 0.0
    begin = time.time()
    sh('mkdir -p "%s" && cd "%s/dataset/d1" && find . -type f -print0 | xargs -0 -P 16 -I{} cp --parents {} "%s"' % (target, DRIVE_REPO, target))
    return float(sh('du -sm "%s" | cut -f1' % target)) / 1000, time.time() - begin


weights = ["DINO/dinov2_vits14_timm.safetensors"]
for name in COLAB_MATCHERS:
    if name == "lightglue":
        weights.append("MINIMA/minima_lightglue.pth")
    if name == "loftr_minima":
        weights.append("MINIMA/minima_loftr.ckpt")
    if name == "roma_minima":
        weights += ["MINIMA/minima_roma.pth", "RoMa/dinov2_vitl14_pretrain.pth"]
weights = sorted(set(weights))

start = time.time()
with ThreadPoolExecutor(COPY_WORKERS) as pool:
    scene_jobs = [pool.submit(get_scene, scene) for scene in SCENES]
    weight_jobs = [pool.submit(copy_one, DRIVE_REPO + "/dataset/NUDT/checkpoint/" + w, LOCAL_REPO + "/dataset/NUDT/checkpoint/" + w) for w in weights]
    d1_job = pool.submit(get_d1)
    for job in scene_jobs:
        scene, gigabytes, copy_seconds, unzip_seconds = job.result()
        print("map scene %s: %.1f GB copied in %.0f s (%.0f MB/s), unpacked in %.0f s" % (scene, gigabytes, copy_seconds, 1000 * gigabytes / max(copy_seconds, 1e-9), unzip_seconds))
    for name, job in zip(weights, weight_jobs):
        gigabytes, seconds = job.result()
        print("weights %s: %.2f GB in %.0f s" % (name, gigabytes, seconds))
    gigabytes, seconds = d1_job.result()
    print("d1 photos: %.2f GB in %.0f s" % (gigabytes, seconds))
print("data on the local disk after %.0f s in total" % (time.time() - start))

os.makedirs(LOCAL_REPO + "/dataset/NUDT/Reference_map", exist_ok=True)       # find_nudt_dir only checks that these two folders exist
os.makedirs(LOCAL_REPO + "/dataset/NUDT/Thermal-UAV", exist_ok=True)
assert os.path.exists(LOCAL_REPO + "/dataset/d1/images/metadata.csv"), "d1 photos missing after the copy"
for scene in SCENES:
    assert os.path.exists(LOCAL_REPO + "/dataset/Dataset_Official_2/" + scene + "/BAND.tif"), "map scene missing: " + scene
print(sh("df -h /content | tail -1"))

# ---------------- 3. Python 3.10 environment ----------------
ENV = "/content/env310"
PY = ENV + "/bin/python"
PIP = ENV + "/bin/pip"
CHECK = "import numpy, cv2, torch, rasterio, pyproj, timm, kornia, lightglue, romatch, ipykernel; print(numpy.__version__, torch.__version__)"


def env_is_ready():
    return os.path.exists(PY) and subprocess.run([PY, "-c", CHECK], capture_output=True).returncode == 0


if env_is_ready():
    print("Python 3.10 environment already there:", sh(PY + " -c '" + CHECK + "'"))
else:
    print("building the Python 3.10 environment (about 8 to 12 minutes the first time) ...")
    sh("apt-get update -y -qq && apt-get install -y -qq software-properties-common")
    sh("add-apt-repository -y ppa:deadsnakes/ppa && apt-get update -y -qq")
    sh("apt-get install -y -qq python3.10 python3.10-venv python3.10-dev")
    sh("rm -rf " + ENV + " && python3.10 -m venv " + ENV)
    sh(PIP + " install -q --upgrade pip setuptools wheel")
    sh("echo numpy==1.23.5 > /content/constraints.txt")
    sh(PIP + " install -q -c /content/constraints.txt numpy==1.23.5")
    sh(PIP + " install -q -c /content/constraints.txt torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu126")
    sh(PIP + " install -q -c /content/constraints.txt pandas==1.5.3 matplotlib==3.7.5 opencv-python-headless==4.8.1.78 timm==0.9.2 huggingface_hub==0.17.3 "
             "rasterio==1.3.9 pyproj==3.6.1 kornia==0.7.1 einops loguru tqdm h5py scipy Pillow safetensors "
             "ipykernel==6.29.5 ipython==8.12.3 traitlets==5.14.3 jupyter_client==8.6.1")
    sh(PIP + " install -q --no-deps git+https://github.com/cvg/LightGlue.git")
    sh(PIP + " install -q --no-deps romatch==0.1.2")
    print("built:", sh(PY + " -c '" + CHECK + "'"))
sh(PY + " -m ipykernel install --user --name=env310 --display-name 'Python 3.10 (GeoLoc venv)'")
spec_file = os.path.expanduser("~/.local/share/jupyter/kernels/env310/kernel.json")
spec = json.load(open(spec_file))
spec["argv"] = [a for a in spec["argv"] if not a.startswith("--IPKernelApp")] + ["--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel", "--InteractiveShellApp.extensions=[]", "--InteractiveShellApp.reraise_ipython_extension_failures=False"]
json.dump(spec, open(spec_file, "w"))

# ---------------- 4. one Python 3.10 kernel that stays alive ----------------
from jupyter_client.manager import KernelManager
from IPython import get_ipython
from IPython.display import display, Image, HTML

if "km310" in globals() and km310.is_alive():
    km310.shutdown_kernel(now=True)                         # start fresh when this cell is run again
km310 = KernelManager(kernel_name="env310")
klog = open("/content/kernel310.log", "w")
clean_env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH" and not k.startswith("COLAB_")}   # Colab settings that Python 3.10 cannot load
clean_env.update(MPLBACKEND="module://matplotlib_inline.backend_inline", IPYTHONDIR="/content/ipy310", JUPYTER_NO_CONFIG="1")
km310.start_kernel(env=clean_env, cwd=LOCAL_REPO, stderr=klog, stdout=klog)     # cwd = the repo: the notebook finds its root from there
kc310 = km310.client()
kc310.start_channels()
kc310.wait_for_ready(timeout=180)
ANSI = re.compile(r"\x1b\[[0-9;]*m")


def run310(code):
    # sends one cell to the Python 3.10 kernel and shows what comes back (text, pictures, tables, errors)
    if not km310.is_alive():
        km310.restart_kernel(now=True)
        kc310.wait_for_ready(timeout=180)
        print("NOTE: the Python 3.10 kernel had died and was restarted: all variables are gone, run the cells above again.")
    msg_id = kc310.execute(code)
    error = None
    while True:
        try:
            msg = kc310.get_iopub_msg(timeout=5)
        except queue.Empty:
            continue
        except KeyboardInterrupt:
            km310.interrupt_kernel()
            continue
        if msg["parent_header"].get("msg_id") != msg_id:
            continue
        kind, content = msg["msg_type"], msg["content"]
        if kind == "stream":
            print(content["text"], end="", file=sys.stderr if content["name"] == "stderr" else sys.stdout)
            sys.stdout.flush()
        elif kind in ("display_data", "execute_result"):
            data = content["data"]
            if "image/png" in data:
                display(Image(data=base64.b64decode(data["image/png"])))
            elif "text/html" in data:
                display(HTML(data["text/html"]))
            elif "text/plain" in data:
                print(data["text/plain"])
        elif kind == "error":
            error = content
        elif kind == "status" and content["execution_state"] == "idle":
            break
    if error is not None:
        print(ANSI.sub("", "\n".join(error["traceback"])), file=sys.stderr)
        raise RuntimeError(error["ename"] + ": " + error["evalue"])


def send_cell_to_310(lines):
    code = "".join(lines)
    first = code.lstrip().split("\n")[0] if code.strip() else ""
    if first.startswith(("%", "!", "#main")) or "run310(" in code or "km310" in code:
        return lines                                        # shell / magic cells, #main cells and this cell run in the normal kernel
    if not code.strip():
        return lines
    return ["run310(%r)\n" % code]


ip = get_ipython()
ip.input_transformers_cleanup[:] = [f for f in ip.input_transformers_cleanup if getattr(f, "__name__", "") != "send_cell_to_310"]
ip.input_transformers_cleanup.append(send_cell_to_310)
run310("import os, sys, platform, torch; print('cells now run in Python', platform.python_version(), '| torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| folder', os.getcwd())")
print("Done. To run a cell in the normal Colab Python instead, start it with the line  #main")

# Part 1 · Settings

In [ ]:
import os, sys, math, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
DATA = "official"           # "official" (d1 photos) or "nudt" (Changsha test photos)
N_PHOTOS = 500              # photos loaded for the data set (spread over the flights)
N_EVAL = 100                # photos evaluated
FIRST_PHOTO = 0             # the first photo of the window (photos FIRST_PHOTO ... FIRST_PHOTO + N_EVAL - 1): use another window to test a setting on photos it was not tuned on
COARSE = "fused"            # the coarse method that goes on to the fine stage: "fused" or "edge_zncc". The other one is evaluated for stages 1 to 3 only
MATCHERS = ["loftr_minima"] # fine matchers to evaluate, e.g. ["roma_minima", "loftr_minima", "lightglue"]. The first one is the baseline in all tables
K_MAX = 100                 # peaks kept per photo and sent through the fine stage once (every K and M of the sweeps is taken from these)
FOUND_SIGMA = 4.0           # a place counts as found when it is within this many sigma of the truth
STRICT_SIGMA = 1.0          # the strict version, shown next to it

# the baseline setting for the stage tables (the sweeps in Part 8 vary these)
USE_SEMANTIC_LAYER = True   # DINO re-orders the K shortlist and the best M go to the fine stage; False = the first M of the coarse order
K_BASE = 100                # K: places the coarse search returns
M_BASE = 25                 # M <= K: places that count as sent to the fine stage
WEIGHT_BASE = 100.0         # DINO weight in the re-ordering: z(coarse) + WEIGHT * z(DINO); 100 = DINO decides inside the K, 0 = coarse order
MIN_INLIERS = 8             # answer bar: answer when the chosen place has at least this many inliers
GAMMA = 1.0                 # choose by  inliers + GAMMA * (DINO z-score);  0 = inliers only
RESCUE_INLIERS = None       # e.g. 6: also answer with this many inliers (instead of MIN_INLIERS) ...
RESCUE_Z = None             # ... when the DINO z-score of the chosen place is at least this (e.g. 2.0)
RANSAC_PIXELS = None        # inlier distance in pixels (None = 3.0 from settings)
MATCHER_CONFIDENCE = 0.2    # LoFTR only: matches below this confidence are dropped before RANSAC (0.2 = kornia's default; lower lets weak, possibly false matches through)
LOFTR_COARSE_THRESHOLD = 0.2   # LoFTR only: its own cut on the coarse match probability (None = 0.2)

# the values the sweeps in Part 8 go through: one logical range per hyperparameter (the baseline above should lie inside it)
K_LIST = [5, 10, 25, 50, 100, 150, 200]          # places the coarse search returns (values above K_MAX are skipped)
M_LIST = [1, 3, 5, 10, 15, 25, 50]               # places sent to the fine stage (1 = trust the first place; M > K is skipped)
WEIGHT_LIST = [0.0, 0.3, 1.0, 3.0, 10.0, 100.0]  # DINO weight in the re-ordering: 0 = coarse order ... 100 = DINO decides
BAR_MIN = 4                                      # the answer bar is swept from BAR_MIN to BAR_MAX in steps of BAR_STEP (inliers the chosen place needs)
BAR_MAX = 16
BAR_STEP = 3
BAR_LOW = []                                     # extra low bars (not used here: they are not safe with this matcher)
BAR_LIST = sorted(set(BAR_LOW + list(range(BAR_MIN, BAR_MAX + 1, BAR_STEP))))      # 1, 2, 3, 4, 7, 10, 13, 16
GAMMA_LIST = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]     # weight of the DINO z-score when choosing the candidate (0 = inliers only)
RESCUE_LIST = [None, (4, 2.0), (6, 2.0), (8, 2.5)]   # (inliers, DINO z): also answer with fewer inliers when DINO is sure

# What changes the data itself (MAP, COARSE, MATCHERS, N_EVAL) needs a new run of the notebook.

N_SAMPLES = 4               # sample figures per kind of failure (Part 9)
SEED = 451
plt.rcParams["figure.dpi"] = 90

assert FIRST_PHOTO >= 0, "FIRST_PHOTO must be 0 or more"
assert FOUND_SIGMA > STRICT_SIGMA > 0, "need FOUND_SIGMA > STRICT_SIGMA > 0"
assert len(MATCHERS) >= 1, "give at least one matcher"
assert K_MAX >= K_BASE, "K_MAX must be at least K_BASE"
K_LIST = [k for k in K_LIST if k <= K_MAX]
assert 1 <= M_BASE <= K_BASE, "need 1 <= M_BASE <= K_BASE"
assert (RESCUE_INLIERS is None) == (RESCUE_Z is None), "give RESCUE_INLIERS and RESCUE_Z together"
assert 1 <= BAR_MIN <= BAR_MAX and BAR_STEP >= 1, "need 1 <= BAR_MIN <= BAR_MAX and BAR_STEP >= 1"
assert LOFTR_COARSE_THRESHOLD is None or 0 < LOFTR_COARSE_THRESHOLD < 1, "LOFTR_COARSE_THRESHOLD must be between 0 and 1 (or None)"


# Part 2 · Set up once

The same pieces as `single_photo_pipeline.py`: photos and camera, the map at all zoom levels, the feature families, the trained fusion model, the DINO network and the first matcher. Here the pipeline object is used as a **toolbox** (its parts are called one by one), not as a single call, because we want to see what every stage produced.

In [ ]:
from single_photo_pipeline import SinglePhotoPipeline      # first import: it saves the development-set settings before NUDT changes them
import cv2
import torch
from header import settings
from header.settings import DEVICE
from train import config
from src.header.scoring import points_for_error, final_score
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, map_piece
from image_processing.edge_features import query_edge_features
from image_processing.thermal import load_thermal_frame, normalise_frame
from train.coarse_methods.inputs import make_inputs
from train.coarse_methods.single_map import make_single_map_method
from semantic_rerank.dino_rerank import shortlist_peaks, dino_candidate_scores, rerank_peaks
from semantic_rerank.decision import shortlist_z, choose_candidate, decide_with_dino
from fine_matching.second_stage_any import run_fine_stage

pipeline = SinglePhotoPipeline(DATA, COARSE, semantic_layer=True, semantic_decision=True, k_peaks=K_MAX, m_fine=K_MAX,
                               matcher=MATCHERS[0], min_inliers=MIN_INLIERS, ransac_pixels=RANSAC_PIXELS,
                               matcher_confidence=MATCHER_CONFIDENCE, loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD)
start = time.time()
pipeline.setup(N_PHOTOS, SEED)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)

test_set = pipeline.test_set
camera = pipeline.camera
level_images = pipeline.level_images

# the coarse methods that are evaluated: the fused one (or edge_zncc if there is no saved fusion model) and edge_zncc alone
PRIMARY = pipeline.coarse_name
METHODS = {"edge_zncc": make_single_map_method("edge_zncc")}
if PRIMARY != "edge_zncc":
    METHODS[PRIMARY] = pipeline.coarse_method
print("coarse methods evaluated:", list(METHODS), "| the fine stage runs on:", PRIMARY)
print("maps inside the fusion:", pipeline.names)

# Part 3 · Pass A: photo, score maps, peaks and DINO scores (no matcher yet)

For every photo, without the matcher:

1. **Render** the north-up canvas. Keep its valid share, contrast and edge strength (stage 1).
2. **Score maps.** For each map in the fusion and for each coarse method, find where in the 5 km disc the best score lies relative to the truth: `truth_stats` gives the global maximum's distance from the truth (in sigma), and the best score within 4 sigma of the truth against the rest of the disc (a z-score and a margin).
3. **Peaks.** For each coarse method, the K_MAX best peaks, their error in sigma, and the DINO scores of those places.

The truth is used only to judge. Nothing the pipeline does depends on it.

In [ ]:
def errors_in_sigma(peaks, index):
    assert peaks.ndim == 2 and peaks.shape[1] >= 3, "peaks must be (n, >= 3)"
    truth = test_set.truths[index]
    sigma = test_set.sigmas[index]
    errors = np.full(len(peaks), 1e9)
    for k in range(len(peaks)):
        if peaks[k, 1] < 1e8:
            errors[k] = math.hypot(peaks[k, 1] - truth[0], peaks[k, 2] - truth[1]) / sigma
    return errors


def truth_stats(score_map, usable, geometry, truth, sigma):
    # What does this score map say about the true place?
    #   argmax_err  distance (in sigma) of the highest score of the whole disc from the truth
    #   near_z      z-score (over the disc) of the best score within FOUND_SIGMA of the truth
    #   margin      best score near the truth minus best score elsewhere (positive = the truth area wins)
    assert score_map.shape == usable.shape == geometry["inside"].shape, "score map, usable mask and grid must have the same shape"
    assert sigma > 0, "sigma must be positive"
    empty = dict(argmax_err=np.nan, near_z=np.nan, margin=np.nan)
    if int(usable.sum()) < 10:
        return empty
    x_m = geometry["x_m"]
    y_m = geometry["y_m"]
    distance_sq = (x_m[None, :] - truth[0]) ** 2 + (y_m[:, None] - truth[1]) ** 2
    values = score_map[usable]
    mean = float(values.mean())
    spread = float(values.std()) + 1e-9

    masked = score_map.clone()
    masked[~usable] = -1e9
    best_flat = int(masked.reshape(-1).argmax())
    row = best_flat // score_map.shape[1]
    col = best_flat % score_map.shape[1]
    argmax_err = math.hypot(float(x_m[col]) - truth[0], float(y_m[row]) - truth[1]) / sigma

    near = usable & (distance_sq <= (FOUND_SIGMA * sigma) ** 2)
    far = usable & (distance_sq > (FOUND_SIGMA * sigma) ** 2)
    if int(near.sum()) == 0 or int(far.sum()) == 0:
        return dict(argmax_err=argmax_err, near_z=np.nan, margin=np.nan)
    near_best = float(masked[near].max())
    far_best = float(masked[far].max())
    return dict(argmax_err=argmax_err, near_z=(near_best - mean) / spread, margin=near_best - far_best)

In [ ]:
records = []
failures = []
assert FIRST_PHOTO < len(test_set.items), "FIRST_PHOTO %d is beyond the %d photos" % (FIRST_PHOTO, len(test_set.items))
count = min(N_EVAL, len(test_set.items) - FIRST_PHOTO)
start = time.time()
for index in range(FIRST_PHOTO, FIRST_PHOTO + count):
    try:
        item = test_set.items[index]
        truth = test_set.truths[index]
        sigma = test_set.sigmas[index]
        cv2.setRNGSeed(settings.SEED)
        level = level_for_height(item["rel_height"], camera)
        canvas, valid = render_query(item, camera, level)
        assert valid.any(), "the canvas has no valid pixel"

        query_features, query_cells = query_edge_features(canvas, valid)
        cells_ok = np.asarray(query_cells) > 0
        edge_strength = 0.0
        if cells_ok.any():
            edge_strength = float(query_features.sum(axis=0)[cells_ok].mean())

        inputs = make_inputs(canvas, valid, level, test_set.priors[index], pipeline.families, pipeline.names)
        geometry = inputs["geometry"]
        record = dict(index=index, height=item["rel_height"], level=level, truth=truth, sigma=sigma,
                      canvas=canvas, valid=valid, valid_frac=float(valid.mean()), contrast=float(canvas[valid].std()),
                      edge_strength=edge_strength, maps={}, methods={})

        for name in pipeline.names:
            raw = inputs["maps"][name]
            usable = inputs["usable"] & (raw > -1)
            record["maps"][name] = truth_stats(raw, usable, geometry, truth, sigma)

        for method_name, method in METHODS.items():
            score_map = method(inputs)
            usable = geometry["inside"] & (score_map > -1)
            stats = truth_stats(score_map, usable, geometry, truth, sigma)
            short = shortlist_peaks(score_map, geometry, K_MAX)
            pooled, dense = dino_candidate_scores(pipeline.encoder, canvas, valid, level_images, level, short)
            record["methods"][method_name] = dict(stats=stats, short=short, dense=dense, pooled=pooled, err=errors_in_sigma(short, index))
        records.append(record)
        del inputs
    except Exception as error:
        failures.append(dict(photo=index, error="%s: %s" % (type(error).__name__, str(error)[:120])))
        print("photo %d failed: %s: %s" % (index, type(error).__name__, str(error)[:120]))
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if (index - FIRST_PHOTO + 1) % 20 == 0:
        print("pass A: photo %d/%d, %.0f s" % (index - FIRST_PHOTO + 1, count, time.time() - start))

record_by_index = {record["index"]: record for record in records}
print("pass A done: %d photos in %.0f s (%d failed)" % (len(records), time.time() - start, len(failures)))
assert len(records) > 0, "no photo went through pass A"

# Part 4 · Pass B: the fine matcher on every peak, once per matcher

The matcher runs on all K_MAX peaks of the **primary coarse method**, in the order the coarse search gave them. Inliers and shifts are stored per peak. Because every candidate is judged on its own, any later choice of K, M, DINO weight or answer bar only picks rows out of these numbers. For several matchers the models are loaded one after the other, so only one is in the GPU at a time.

For `loftr_minima` the raw matches of every candidate (positions and confidence) are kept as well, so Part 8.4 can change the confidence cut and the RANSAC distance without running the network again.

In [ ]:
def load_fine_matcher(name):
    assert name in ("lightglue", "loftr_minima", "roma_minima"), "unknown matcher %r" % (name,)
    checkpoints = config.find_nudt_dir() / "checkpoint"
    if name == "loftr_minima":
        from fine_matching.loftr_matcher import load_loftr_matcher
        return load_loftr_matcher(checkpoints / "MINIMA" / "minima_loftr.ckpt", MATCHER_CONFIDENCE, 1.0, LOFTR_COARSE_THRESHOLD)
    if name == "roma_minima":
        from fine_matching.roma_matcher import load_roma_matcher
        return load_roma_matcher(checkpoints / "MINIMA" / "minima_roma.pth", checkpoints / "RoMa" / "dinov2_vitl14_pretrain.pth")
    from fine_matching.matcher import load_matcher
    settings.MINIMA_FILE = checkpoints / "MINIMA" / "minima_lightglue.pth"
    return load_matcher()


from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches

fine = {}
fine_seconds = {}
raw_matches = {}      # LoFTR only: the raw matches of every candidate, so that the confidence cut and the RANSAC distance can be swept without the network
for name in MATCHERS:
    if name == MATCHERS[0]:
        matcher = pipeline.matcher
    else:
        matcher = load_fine_matcher(name)
    stored = {}
    start = time.time()
    for position, record in enumerate(records):
        short = record["methods"][PRIMARY]["short"]
        if name == "loftr_minima":
            cv2.setRNGSeed(settings.SEED)
            kept = candidate_matches(matcher, record["canvas"], record["valid"], record["level"], short, level_images)
            raw_matches.setdefault(name, {})[record["index"]] = kept
            stage2 = stage2_from_matches(kept, record["level"], matcher.min_confidence)
        else:
            stage2 = run_fine_stage(matcher, record["canvas"], record["valid"], record["level"], short, level_images)
        assert len(stage2["inliers"]) == len(short), "the fine stage must return one inlier count per peak"
        stored[record["index"]] = dict(inliers=np.asarray(stage2["inliers"], dtype=float), shifts=np.asarray(stage2["shifts"], dtype=float))
        if (position + 1) % 20 == 0:
            print("pass B (%s): photo %d/%d, %.0f s" % (name, position + 1, len(records), time.time() - start))
    fine[name] = stored
    fine_seconds[name] = time.time() - start
    print("pass B (%s) done in %.0f s: %.2f s per photo for %d candidates" % (name, fine_seconds[name], fine_seconds[name] / len(records), K_MAX))
    if name != MATCHERS[0]:
        del matcher
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

# Part 5 · Scoring tools (no GPU, no matcher)

Everything below works on the stored numbers.

* `ordered_errors` : for a setting (K, M, layer, weight) the order in which the peaks go to the fine stage, and the error of each. With the layer on this is the same re-ordering as `semantic_layer` (`rerank_peaks` over the first K peaks); with the layer off it is the first M of the coarse order. The position in the shortlist is carried in column 3 of a copy of the peaks, because `rerank_peaks` does not use that column.
* `answer_for` : the answer for one photo under one setting (choose the candidate, decide, score), the same rules as `single_photo_pipeline.py`.
* `evaluate` : `answer_for` for every photo, as a table.
* `summarise` : one line of numbers for a table of settings.

In [ ]:
def ordered_errors(record, method_name, K, M, layer, weight):
    assert 1 <= M <= K <= K_MAX, "need 1 <= M <= K <= K_MAX, got M=%s K=%s" % (M, K)
    assert weight >= 0, "weight must be >= 0"
    data = record["methods"][method_name]
    if not layer:
        order = np.arange(M)
    else:
        sub = data["short"][:K].copy()
        sub[:, 3] = np.arange(K)                 # column 3 carries the position in the shortlist
        order = rerank_peaks(sub, data["dense"][:K], weight, keep=M)[:, 3].astype(int)
    return order, data["err"][order]


def answer_for(record, fine_one, cfg):
    assert set(cfg) >= {"K", "M", "layer", "weight", "gamma", "rescue", "bar"}, "cfg needs K, M, layer, weight, gamma, rescue, bar"
    data = record["methods"][PRIMARY]
    order, errors = ordered_errors(record, PRIMARY, cfg["K"], cfg["M"], cfg["layer"], cfg["weight"])
    peaks = data["short"][order]
    z_short = shortlist_z(data["dense"][:cfg["K"]]) if cfg["layer"] else np.zeros(cfg["K"])
    z = z_short[order]
    stage2 = dict(inliers=fine_one["inliers"][order], shifts=fine_one["shifts"][order])
    gamma = cfg["gamma"] if cfg["layer"] else 0.0
    rescue = cfg["rescue"] if cfg["layer"] else None
    x, y, inliers, chosen, z_chosen = choose_candidate(peaks, stage2, z, gamma)
    if rescue is None:
        answered = decide_with_dino(inliers, z_chosen, min_inliers=cfg["bar"])
    else:
        answered = decide_with_dino(inliers, z_chosen, rescue[0], rescue[1], min_inliers=cfg["bar"])
    truth = record["truth"]
    error_m = math.hypot(x - truth[0], y - truth[1])
    earned = points_for_error(error_m, record["height"], camera.fx)
    all_errors = data["err"][:cfg["K"]]
    row = dict(photo=record["index"], in_K=bool((all_errors <= FOUND_SIGMA).any()), in_M=bool((errors <= FOUND_SIGMA).any()),
               in_M_strict=bool((errors <= STRICT_SIGMA).any()), top1_ok=bool(errors[0] <= FOUND_SIGMA),
               chosen_right=bool(errors[chosen] <= FOUND_SIGMA), answered=bool(answered), inliers=float(inliers),
               z_chosen=float(z_chosen), err_sigma=error_m / record["sigma"], would_earn=earned,
               points=earned if answered else 0,
               chosen_rank=int(order[chosen]), top_first=bool(chosen == 0),               # position of the chosen place in the coarse order; was it the first place after the DINO order?
               dx_m=float(x - truth[0]), dy_m=float(y - truth[1]), err_m=error_m, height=float(record["height"]))   # signed east / north error, for the bias analysis (S7)
    details = dict(order=order, errors=errors, chosen=chosen, x=x, y=y)
    return row, details


def evaluate(cfg, matcher_name):
    assert matcher_name in fine, "matcher %r was not run in pass B" % (matcher_name,)
    rows = []
    for record in records:
        row, details = answer_for(record, fine[matcher_name][record["index"]], cfg)
        rows.append(row)
    return pd.DataFrame(rows)


def summarise(table):
    assert len(table) > 1, "need at least two photos"
    half = len(table) // 2
    answered = table["answered"]
    answered_ok = answered & (table["err_sigma"] <= FOUND_SIGMA)
    result = {}
    result["score (/60)"] = final_score(list(table["points"]))
    result["first half"] = final_score(list(table["points"][:half]))
    result["second half"] = final_score(list(table["points"][half:]))
    result["answered"] = answered.mean()
    result["target in K"] = table["in_K"].mean()
    result["target in M"] = table["in_M"].mean()
    result["top-1 ok"] = table["top1_ok"].mean()
    if table["in_M"].any():
        result["right chosen | target in M"] = table["chosen_right"][table["in_M"]].mean()
    else:
        result["right chosen | target in M"] = np.nan
    result["answered and ok"] = answered_ok.mean()
    result["wrong answers"] = int((table["points"] < 0).sum())
    result["right but skipped"] = int(((~answered) & (table["err_sigma"] <= FOUND_SIGMA)).sum())
    return result


BASE = dict(K=K_BASE, M=M_BASE, layer=USE_SEMANTIC_LAYER, weight=WEIGHT_BASE, gamma=GAMMA,
            rescue=None if RESCUE_INLIERS is None else (RESCUE_INLIERS, RESCUE_Z), bar=MIN_INLIERS)
base = {name: evaluate(BASE, name) for name in MATCHERS}
print("baseline setting:", BASE)
display(pd.DataFrame({name: summarise(base[name]) for name in MATCHERS}).T.round(3))

# Part 6 · The stages one by one

## 6.1 Stage 1 · Image processing

Two questions: is the canvas a usable picture (valid share, contrast, edge strength), and does it look like the map **where it truly is**? The second one is measured on the score maps: if the true place does not stand out in a map, no search on top of that map can find it. For each map:

* **max at target (4σ / 1σ)**: the highest score of the whole disc lies within 4 (1) sigma of the truth. This is the strongest test: the map alone would give the right answer.
* **z near truth**: how many standard deviations above the disc's mean the best score near the truth is. Above 3 means the truth area is clearly visible in the map.
* **margin > 0**: the truth area holds a higher score than everything else.

In [ ]:
photo_rows = []
for record in records:
    row = dict(photo=record["index"], height=record["height"], level=record["level"], valid_frac=record["valid_frac"],
               contrast=record["contrast"], edge_strength=record["edge_strength"])
    row["near_z (%s)" % PRIMARY] = record["methods"][PRIMARY]["stats"]["near_z"]
    photo_rows.append(row)
image_table = pd.DataFrame(photo_rows)
display(image_table.drop(columns=["photo"]).describe().T[["mean", "min", "25%", "50%", "75%", "max"]].round(3))

all_map_names = list(pipeline.names)
for method_name in METHODS:
    if method_name not in all_map_names:
        all_map_names.append(method_name)
map_rows = []
for name in all_map_names:
    stats = []
    for record in records:
        if name in record["maps"]:
            stats.append(record["maps"][name])
        else:
            stats.append(record["methods"][name]["stats"])
    argmax_err = np.array([s["argmax_err"] for s in stats])
    near_z = np.array([s["near_z"] for s in stats])
    margin = np.array([s["margin"] for s in stats])
    map_rows.append({"map": name,
                     "max at target (4 sigma)": np.nanmean(argmax_err <= FOUND_SIGMA),
                     "max at target (1 sigma)": np.nanmean(argmax_err <= STRICT_SIGMA),
                     "median z near truth": np.nanmedian(near_z),
                     "z near truth > 3": np.nanmean(near_z > 3),
                     "margin > 0": np.nanmean(margin > 0)})
map_table = pd.DataFrame(map_rows)
display(map_table.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
hit = np.array([record["methods"][PRIMARY]["stats"]["argmax_err"] <= FOUND_SIGMA for record in records])
for axis, column, label in ((axes[0], "valid_frac", "valid share of the canvas"), (axes[1], "contrast", "contrast of the canvas"), (axes[2], "edge_strength", "edge strength")):
    axis.scatter(image_table[column], image_table["near_z (%s)" % PRIMARY], c=["tab:green" if h else "tab:gray" for h in hit], s=16)
    axis.set_xlabel(label)
    axis.set_ylabel("z near truth (%s)" % PRIMARY)
axes[0].set_title("green = the %s map's maximum is at the target" % PRIMARY, fontsize=9)
plt.tight_layout()
plt.show()

image_table["valid_group"] = pd.qcut(image_table["valid_frac"].rank(method="first"), 3, labels=["low valid share", "middle", "high valid share"])
image_table["map_hit"] = hit
print("share of photos whose %s maximum is at the target, by valid share of the canvas:" % PRIMARY)
display(image_table.groupby("valid_group", observed=True)["map_hit"].agg(["mean", "count"]).round(3))

## 6.2 Stage 2 · Coarse search: is the target among the chosen peaks?

**Recall at K** = the share of photos with a peak within 4 sigma (1 sigma) of the truth among the first K peaks. Recall at K = 1 is the coarse top-1. The gap between recall at 1 and recall at 50 is what a better judge of the shortlist (stage 3) could still win; the gap between recall at 50 and 100 % is lost for good at this stage.

In [ ]:
recall_ks = [k for k in (1, 3, 5, 10, 25, 50) if k <= K_MAX]
rows = []
curves = {}
for method_name in METHODS:
    for sigma_limit, label in ((FOUND_SIGMA, "%g sigma" % FOUND_SIGMA), (STRICT_SIGMA, "%g sigma" % STRICT_SIGMA)):
        row = {"method": method_name, "found within": label}
        values = []
        for k in recall_ks:
            hits = [(record["methods"][method_name]["err"][:k] <= sigma_limit).any() for record in records]
            row["recall@%d" % k] = np.mean(hits)
            values.append(np.mean(hits))
        rows.append(row)
        curves[(method_name, label)] = values
display(pd.DataFrame(rows).round(3))

first_right = []
for record in records:
    errors = record["methods"][PRIMARY]["err"]
    right = np.where(errors <= FOUND_SIGMA)[0]
    if len(right) > 0:
        first_right.append(int(right[0]) + 1)
print("%s: the target is in the first %d peaks for %d of %d photos; rank of the first right peak: median %.0f, mean %.1f" % (
    PRIMARY, K_MAX, len(first_right), len(records), np.median(first_right) if first_right else float("nan"), np.mean(first_right) if first_right else float("nan")))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for (method_name, label), values in curves.items():
    axes[0].plot(recall_ks, values, "o-", label="%s, %s" % (method_name, label))
axes[0].set_xscale("log")
axes[0].set_xlabel("K (peaks looked at)")
axes[0].set_ylabel("recall")
axes[0].set_title("target in the first K peaks")
axes[0].legend(fontsize=8)
axes[1].hist(first_right, bins=range(1, K_MAX + 2), color="tab:blue")
axes[1].set_xlabel("rank of the first right peak (%s)" % PRIMARY)
axes[1].set_ylabel("photos")
axes[1].set_title("where the first right peak sits")
plt.tight_layout()
plt.show()

misses = [record for record in records if not (record["methods"][PRIMARY]["err"][:K_MAX] <= FOUND_SIGMA).any()]
missed_photos = set(record["index"] for record in misses)
hits_list = [record for record in records if record["index"] not in missed_photos]
if len(misses) > 0 and len(hits_list) > 0:
    print("z of the best score near the truth: photos where the target is NOT in the %d peaks: median %.2f | photos where it is: median %.2f" % (
        K_MAX, np.nanmedian([record["methods"][PRIMARY]["stats"]["near_z"] for record in misses]),
        np.nanmedian([record["methods"][PRIMARY]["stats"]["near_z"] for record in hits_list])))
    print("(a low value means the map itself did not show the target: a stage 1 / map problem; a high value means the target was visible but another place was ranked higher)")

## 6.3 Stage 3 · Semantic layer (DINO)

Three questions, all on the same K (= `K_BASE`) peaks:

* **Top-1:** is the first peak right more often after DINO re-ordered the K peaks? Paired: how many photos were fixed, how many broken, and a bootstrap 95 % interval of the gain.
* **Recall at M of K:** the fine stage only gets M of the K peaks. How often is the target among them, with the coarse order and with the DINO order?
* **Separation:** within one photo's K peaks, how often does a right peak get a higher DINO score than a wrong one (0.5 = no information, 1.0 = perfect)?

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for method_name in METHODS:
    for sigma_limit, label in ((FOUND_SIGMA, "%g sigma" % FOUND_SIGMA), (STRICT_SIGMA, "%g sigma" % STRICT_SIGMA)):
        before = []
        after = []
        for record in records:
            err_plain = record["methods"][method_name]["err"][:K_BASE]
            order, err_layer = ordered_errors(record, method_name, K_BASE, 1, True, WEIGHT_BASE)
            before.append(err_plain[0] <= sigma_limit)
            after.append(err_layer[0] <= sigma_limit)
        before = np.array(before)
        after = np.array(after)
        difference = after.astype(float) - before.astype(float)
        resamples = []
        for _ in range(2000):
            pick = rng.integers(0, len(difference), len(difference))
            resamples.append(difference[pick].mean())
        low, high = np.percentile(resamples, [2.5, 97.5])
        rows.append({"coarse method": method_name, "top-1 within": label, "before": before.mean(), "after DINO": after.mean(),
                     "gain": difference.mean(), "95% low": low, "95% high": high,
                     "fixed": int((after & ~before).sum()), "broken": int((~after & before).sum())})
display(pd.DataFrame(rows).round(3))

rows = []
for method_name in METHODS:
    for m in [m for m in M_LIST if m <= K_BASE]:
        plain = []
        layered = []
        for record in records:
            err_plain = record["methods"][method_name]["err"][:m]
            order, err_layer = ordered_errors(record, method_name, K_BASE, m, True, WEIGHT_BASE)
            plain.append((err_plain <= FOUND_SIGMA).any())
            layered.append((err_layer <= FOUND_SIGMA).any())
        ceiling = np.mean([(record["methods"][method_name]["err"][:K_BASE] <= FOUND_SIGMA).any() for record in records])
        rows.append({"coarse method": method_name, "M": m, "target in first M (coarse order)": np.mean(plain),
                     "target in best M (DINO order)": np.mean(layered), "target in all K": ceiling})
display(pd.DataFrame(rows).round(3))

rank_before = []
rank_after = []
auc_values = []
for record in records:
    data = record["methods"][PRIMARY]
    err_plain = data["err"][:K_BASE]
    order, err_layer = ordered_errors(record, PRIMARY, K_BASE, K_BASE, True, WEIGHT_BASE)
    right_plain = np.where(err_plain <= FOUND_SIGMA)[0]
    right_layer = np.where(err_layer <= FOUND_SIGMA)[0]
    if len(right_plain) > 0:
        rank_before.append(right_plain[0] + 1)
        rank_after.append(right_layer[0] + 1)
    right = err_plain <= FOUND_SIGMA
    real = data["short"][:K_BASE, 1] < 1e8
    dense = data["dense"][:K_BASE]
    if right.any() and (real & ~right).any():
        good = dense[right & real][:, None]
        bad = dense[real & ~right][None, :]
        auc_values.append(((good > bad).mean() + 0.5 * (good == bad).mean()))
print("%s: rank of the first right peak among K=%d: before DINO median %.0f (mean %.1f) -> after DINO median %.0f (mean %.1f), %d photos" % (
    PRIMARY, K_BASE, np.median(rank_before), np.mean(rank_before), np.median(rank_after), np.mean(rank_after), len(rank_before)))
print("DINO separation inside the K peaks (right peak scored above a wrong one): mean %.3f over %d photos" % (np.mean(auc_values), len(auc_values)))

## 6.4 Stage 4 · Fine matching: if the target is among the M, does the matcher find it?

For the photos where a right candidate is among the M that go to the fine stage:

* **right has most inliers:** the best right candidate has more inliers than the best wrong one (the matcher separates them).
* **right has 0 inliers:** the matcher found nothing at the target (matcher failure, or the shift was rejected).
* **separation:** within a photo, how often a right candidate has more inliers than a wrong one (0.5 = no information).
* **median inliers right / wrong:** the typical best count of right and wrong candidates. The distance between them shows where the answer bar can sit.
* **error after the shift:** how close the corrected position of the chosen right candidate gets (the precision of the matcher).

In [ ]:
rows = []
for name in MATCHERS:
    wins = []
    zero = []
    auc_list = []
    inliers_right = []
    inliers_wrong = []
    for record in records:
        order, errors = ordered_errors(record, PRIMARY, BASE["K"], BASE["M"], BASE["layer"], BASE["weight"])
        inliers = fine[name][record["index"]]["inliers"][order]
        right = errors <= FOUND_SIGMA
        if not right.any():
            continue
        best_right = inliers[right].max()
        inliers_right.append(best_right)
        zero.append(best_right == 0)
        if (~right).any():
            best_wrong = inliers[~right].max()
            inliers_wrong.append(best_wrong)
            wins.append(best_right > best_wrong)
            good = inliers[right][:, None]
            bad = inliers[~right][None, :]
            auc_list.append(((good > bad).mean() + 0.5 * (good == bad).mean()))
    table = base[name]
    chosen_ok = table[table["chosen_right"] & table["answered"]]
    rows.append({"matcher": name, "photos with the target in M": len(inliers_right),
                 "right has most inliers": np.mean(wins), "right has 0 inliers": np.mean(zero),
                 "separation (right > wrong)": np.mean(auc_list),
                 "median best inliers, right": np.median(inliers_right), "median best inliers, wrong": np.median(inliers_wrong),
                 "median error after shift (sigma)": chosen_ok["err_sigma"].median() if len(chosen_ok) > 0 else np.nan,
                 "answered within 1 sigma": (chosen_ok["err_sigma"] <= STRICT_SIGMA).mean() if len(chosen_ok) > 0 else np.nan,
                 "seconds per photo (K_MAX candidates)": fine_seconds[name] / len(records)})
display(pd.DataFrame(rows).round(3))

fig, axes = plt.subplots(1, len(MATCHERS), figsize=(6 * len(MATCHERS), 3.8), squeeze=False)
for position, name in enumerate(MATCHERS):
    right_values = []
    wrong_values = []
    for record in records:
        order, errors = ordered_errors(record, PRIMARY, BASE["K"], BASE["M"], BASE["layer"], BASE["weight"])
        inliers = fine[name][record["index"]]["inliers"][order]
        right_values.extend(list(inliers[errors <= FOUND_SIGMA]))
        wrong_values.extend(list(inliers[errors > FOUND_SIGMA]))
    top = max(max(right_values + [1]), max(wrong_values + [1]))
    bins = np.linspace(0, top, 40)
    axes[0, position].hist(wrong_values, bins=bins, alpha=0.6, color="tab:gray", label="wrong candidates", density=True)
    axes[0, position].hist(right_values, bins=bins, alpha=0.6, color="tab:green", label="right candidates", density=True)
    axes[0, position].axvline(BASE["bar"], color="red", ls="--", label="answer bar")
    axes[0, position].set_yscale("log")
    axes[0, position].set_xlabel("RANSAC inliers of one candidate")
    axes[0, position].set_title("%s: inliers of right and wrong candidates" % name, fontsize=9)
    axes[0, position].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6.5 Stage 5 · Decision: does it answer the right photos?

* **precision:** among the answered photos, the share within 4 sigma (and within 1 sigma).
* **answerable and answered:** among photos whose chosen position is within 4 sigma, the share that was answered.
* **points lost by skipping:** the points the skipped right answers would have earned.
* **points lost to wrong answers:** the points taken away by answers beyond 8 sigma (each −20).
* The sweep over the answer bar (and gamma and the rescue rule) is in Part 8; this part reports the baseline.

In [ ]:
rows = []
for name in MATCHERS:
    table = base[name]
    answered = table[table["answered"]]
    answerable = table[table["err_sigma"] <= FOUND_SIGMA]
    skipped_right = table[(~table["answered"]) & (table["err_sigma"] <= FOUND_SIGMA)]
    wrong = table[table["points"] < 0]
    rows.append({"matcher": name, "answered": table["answered"].mean(),
                 "precision (4 sigma)": (answered["err_sigma"] <= FOUND_SIGMA).mean() if len(answered) > 0 else np.nan,
                 "precision (1 sigma)": (answered["err_sigma"] <= STRICT_SIGMA).mean() if len(answered) > 0 else np.nan,
                 "answerable photos": len(answerable), "answerable and answered": answerable["answered"].mean() if len(answerable) > 0 else np.nan,
                 "points lost by skipping (/photos)": float(skipped_right["would_earn"].sum()) / len(table),
                 "wrong answers": len(wrong), "points lost to wrong answers (/photos)": 20.0 * len(wrong) / len(table)})
display(pd.DataFrame(rows).round(3))
print("points lost are shown as points per photo: multiply by 6 for the effect on the score out of 60")

## 6.6 Stage 6 · End to end: where do the points go?

The funnel follows each photo until the first stage that loses it. The **ceiling** next to each step is the score out of 60 if every photo that is still alive at that step were answered perfectly (+10) and the rest got 0: it shows how much each stage takes away.

In [ ]:
for name in MATCHERS:
    table = base[name]
    total = len(table)
    answered_ok4 = table["answered"] & (table["err_sigma"] <= FOUND_SIGMA)
    answered_ok1 = table["answered"] & (table["err_sigma"] <= STRICT_SIGMA)
    steps = [("all photos", np.ones(total, dtype=bool)),
             ("target in the K peaks (coarse)", table["in_K"].to_numpy()),
             ("target in the M sent to fine (DINO / M)", (table["in_K"] & table["in_M"]).to_numpy()),
             ("right candidate chosen (fine)", (table["in_K"] & table["in_M"] & table["chosen_right"]).to_numpy()),
             ("answered (decision)", (table["in_K"] & table["in_M"] & table["chosen_right"] & table["answered"]).to_numpy()),
             ("... and within 4 sigma", answered_ok4.to_numpy()),
             ("... and within 1 sigma", answered_ok1.to_numpy())]
    rows = []
    previous = total
    for label, alive in steps:
        rows.append({"step": label, "photos": int(alive.sum()), "share": alive.mean(), "lost here": previous - int(alive.sum()),
                     "ceiling score (/60)": 60 * alive.mean()})
        previous = int(alive.sum())
    print("matcher:", name, "| actual score %.2f /60" % final_score(list(table["points"])))
    display(pd.DataFrame(rows).round(3))

    fig, axis = plt.subplots(figsize=(9, 3.4))
    axis.barh([row["step"] for row in rows][::-1], [row["photos"] for row in rows][::-1], color="tab:blue")
    axis.set_xlabel("photos still alive")
    axis.set_title("funnel (%s)" % name, fontsize=9)
    plt.tight_layout()
    plt.show()

main = base[MATCHERS[0]].copy()
main["height"] = [record_by_index[p]["height"] for p in main["photo"]]
main["height group"] = pd.qcut(main["height"].rank(method="first"), 3, labels=["low", "middle", "high"])
print("by flight height (%s):" % MATCHERS[0])
display(main.groupby("height group", observed=True).apply(lambda g: pd.Series({
    "photos": len(g), "score (/60)": final_score(list(g["points"])), "answered": g["answered"].mean(),
    "target in K": g["in_K"].mean(), "target in M": g["in_M"].mean(), "right chosen | in M": g["chosen_right"][g["in_M"]].mean() if g["in_M"].any() else np.nan})).round(3))

# Part 7 · Image processing sample: render and map agreement

The photos where the score maps see the target least clearly (lowest z near the truth, from 6.1) and the photos where they see it best. For each: the north-up canvas, the true map tile and the canvas laid over it. This is the quickest check of the geometry and the cleaned thermal photo.

In [ ]:
def show(axis, picture, title="", cmap="gray", vmin=0, vmax=1):
    axis.imshow(picture, cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(title, fontsize=8)
    axis.set_xticks([])
    axis.set_yticks([])


def show_render(index, label):
    record = record_by_index[index]
    item = test_set.items[index]
    truth = record["truth"]
    true_tile = map_piece(level_images, record["level"], truth[0], truth[1])
    frame = normalise_frame(load_thermal_frame(item))
    blend = np.stack([true_tile, true_tile, true_tile], axis=-1) * 0.6
    blend[..., 0] = np.where(record["valid"], 0.5 * blend[..., 0] + 0.5 * record["canvas"], blend[..., 0])
    fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
    show(axes[0], frame, "photo %d (%s): thermal, cleaned" % (index, label))
    show(axes[1], record["canvas"] * record["valid"], "canvas | valid %.2f | contrast %.2f" % (record["valid_frac"], record["contrast"]))
    show(axes[2], true_tile, "true map tile")
    show(axes[3], np.clip(blend, 0, 1), "canvas (red) over the true tile | z near truth %.1f" % record["methods"][PRIMARY]["stats"]["near_z"])
    plt.tight_layout()
    plt.show()


ranked = sorted(records, key=lambda r: (np.nan_to_num(r["methods"][PRIMARY]["stats"]["near_z"], nan=-99.0)))
print("lowest z near the truth (the target is hardest to see in the map)")
for record in ranked[:N_SAMPLES]:
    show_render(record["index"], "weak")
print("highest z near the truth")
for record in ranked[::-1][:min(2, N_SAMPLES)]:
    show_render(record["index"], "strong")

# Part 8 · Hyperparameter sweeps (from the stored numbers, no new matching)

Every row re-scores the same 100 photos with a different setting. Reading guide: look at **target in K / target in M** to see what a stage allows at most, at **right chosen | target in M** to see what the fine stage and decision do with it, and at the two halves. A setting that is best in one half and not in the other is noise.

## 8.1 K, M and DINO weight

`weight = 0` is the coarse order. `layer off` takes the first M of the coarse order (no DINO).

In [ ]:
rows = []
for K in K_LIST:
    for M in M_LIST:
        if M > K:
            continue
        for weight in WEIGHT_LIST:
            cfg = dict(BASE, K=K, M=M, weight=weight, layer=True)
            row = {"K": K, "M": M, "weight": weight}
            row.update(summarise(evaluate(cfg, MATCHERS[0])))
            rows.append(row)
for M in M_LIST:
    cfg = dict(BASE, K=K_BASE, M=M, layer=False)
    row = {"K": K_BASE, "M": M, "weight": -1}
    row.update(summarise(evaluate(cfg, MATCHERS[0])))
    rows.append(row)
sweep_kmw = pd.DataFrame(rows)
sweep_kmw["weight"] = sweep_kmw["weight"].replace(-1, np.nan)
print("matcher:", MATCHERS[0], "| bar", BASE["bar"], "| gamma", BASE["gamma"], "| rows with weight NaN = layer off")
display(sweep_kmw.sort_values("score (/60)", ascending=False).head(15).round(3).reset_index(drop=True))
print("score by M (rows) and DINO weight (columns), K =", K_BASE)
display(sweep_kmw[(sweep_kmw["K"] == K_BASE) & sweep_kmw["weight"].notna()].pivot_table(index="M", columns="weight", values="score (/60)").round(2))
print("target in M by M (rows) and DINO weight (columns), K =", K_BASE)
display(sweep_kmw[(sweep_kmw["K"] == K_BASE) & sweep_kmw["weight"].notna()].pivot_table(index="M", columns="weight", values="target in M").round(3))
print("layer off (first M of the coarse order):")
display(sweep_kmw[sweep_kmw["weight"].isna()].drop(columns=["weight"]).round(3).reset_index(drop=True))

## 8.2 Answer bar, gamma and rescue rule, per matcher

The bar is the number of inliers the chosen place needs. `gamma` adds the DINO z-score to the inlier count when the candidate is chosen. The rescue rule also answers with fewer inliers when DINO is confident. Everything is evaluated for each matcher in `MATCHERS`, because the inlier scales differ.

In [ ]:
rows = []
for name in MATCHERS:
    for bar in BAR_LIST:
        for gamma in GAMMA_LIST:
            for rescue in RESCUE_LIST:
                cfg = dict(BASE, bar=bar, gamma=gamma, rescue=rescue)
                row = {"matcher": name, "bar": bar, "gamma": gamma, "rescue": str(rescue)}
                row.update(summarise(evaluate(cfg, name)))
                rows.append(row)
sweep_bar = pd.DataFrame(rows)
display(sweep_bar.sort_values("score (/60)", ascending=False).head(15).round(3).reset_index(drop=True))

plain = sweep_bar[(sweep_bar["gamma"] == 0.0) & (sweep_bar["rescue"] == "None")]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
for name in MATCHERS:
    part = plain[plain["matcher"] == name]
    axes[0].plot(part["bar"], part["score (/60)"], "o-", label=name)
    axes[1].plot(part["bar"], part["answered"], "o-", label=name)
    axes[2].plot(part["bar"], part["wrong answers"], "o-", label=name)
axes[0].set_ylabel("score (/60)")
axes[1].set_ylabel("answered")
axes[2].set_ylabel("wrong answers (-20 each)")
for axis in axes:
    axis.set_xlabel("answer bar (inliers)")
    axis.set_xticks(BAR_LIST)
    axis.legend(fontsize=8)
axes[0].set_title("gamma 0, no rescue")
plt.tight_layout()
plt.show()

print("score by gamma (rows) and rescue rule (columns), matcher %s, bar %d" % (MATCHERS[0], BASE["bar"]))
part = sweep_bar[(sweep_bar["matcher"] == MATCHERS[0]) & (sweep_bar["bar"] == BASE["bar"])]
display(part.pivot_table(index="gamma", columns="rescue", values="score (/60)").round(2))

## 8.3 Best setting per matcher, with both halves

The best row of the answer sweep for each matcher next to the baseline. If the best row's gain is not there in both halves, do not use it.

In [ ]:
rows = []
for name in MATCHERS:
    part = sweep_bar[sweep_bar["matcher"] == name].sort_values("score (/60)", ascending=False)
    best = part.iloc[0]
    reference = summarise(base[name])
    rows.append({"matcher": name, "which": "baseline", "bar": BASE["bar"], "gamma": BASE["gamma"], "rescue": str(BASE["rescue"]),
                 "score (/60)": reference["score (/60)"], "first half": reference["first half"], "second half": reference["second half"],
                 "answered": reference["answered"], "wrong answers": reference["wrong answers"]})
    rows.append({"matcher": name, "which": "best of the sweep", "bar": best["bar"], "gamma": best["gamma"], "rescue": best["rescue"],
                 "score (/60)": best["score (/60)"], "first half": best["first half"], "second half": best["second half"],
                 "answered": best["answered"], "wrong answers": best["wrong answers"]})
display(pd.DataFrame(rows).round(3))

# Part 9 · Sample figures: what the failures look like

For each kind of failure (the first stage that lost the photo, from the baseline setting), the worst `N_SAMPLES` photos, plus a few good ones. Each figure shows, left to right and top to bottom: the north-up canvas, the true map tile, the canvas over the true tile, the coarse score map (green star = truth, squares = best peaks); then the error of the first K peaks by coarse rank (green = right), the error of the M peaks after the DINO order, the inliers of those M candidates (green = right place, red line = answer bar), and the map tile of the candidate the pipeline chose.

In [ ]:
def draw_heat_map(axis, score_map, geometry, peaks, index, how_many=5):
    record = record_by_index[index]
    heat = score_map.cpu().numpy().copy()
    inside = geometry["inside"].cpu().numpy()
    heat[~inside | (heat <= -1)] = np.nan
    x_m = geometry["x_m"].cpu().numpy()
    y_m = geometry["y_m"].cpu().numpy()
    axis.imshow(heat, cmap="inferno", vmin=np.nanpercentile(heat, 5), vmax=np.nanmax(heat), extent=(x_m[0], x_m[-1], y_m[-1], y_m[0]))
    axis.add_patch(plt.Circle(test_set.priors[index], settings.DISC_RADIUS, fill=False, color="deepskyblue", lw=1.0))
    axis.plot(record["truth"][0], record["truth"][1], "*", color="lime", markersize=14, markeredgecolor="black")
    for rank in range(min(how_many, len(peaks))):
        if peaks[rank, 1] < 1e8:
            axis.plot(peaks[rank, 1], peaks[rank, 2], "s", color="cyan", markersize=4, markeredgecolor="black")
            axis.annotate(str(rank + 1), (peaks[rank, 1], peaks[rank, 2]), color="white", fontsize=8, xytext=(3, 3), textcoords="offset points")
    axis.set_title("%s score map (star = truth, squares = best peaks)" % PRIMARY, fontsize=8)
    axis.tick_params(labelsize=6)


def draw_sample(index, label):
    record = record_by_index[index]
    row, details = answer_for(record, fine[MATCHERS[0]][index], BASE)
    data = record["methods"][PRIMARY]
    truth = record["truth"]
    level = record["level"]
    true_tile = map_piece(level_images, level, truth[0], truth[1])
    blend = np.stack([true_tile, true_tile, true_tile], axis=-1) * 0.6
    blend[..., 0] = np.where(record["valid"], 0.5 * blend[..., 0] + 0.5 * record["canvas"], blend[..., 0])

    inputs = make_inputs(record["canvas"], record["valid"], level, test_set.priors[index], pipeline.families, pipeline.names)
    score_map = METHODS[PRIMARY](inputs)

    fig = plt.figure(figsize=(17, 8.6))
    grid = fig.add_gridspec(2, 4)
    show(fig.add_subplot(grid[0, 0]), record["canvas"] * record["valid"], "canvas (valid %.2f)" % record["valid_frac"])
    show(fig.add_subplot(grid[0, 1]), true_tile, "true map tile")
    show(fig.add_subplot(grid[0, 2]), np.clip(blend, 0, 1), "canvas (red) over the true tile")
    draw_heat_map(fig.add_subplot(grid[0, 3]), score_map, inputs["geometry"], data["short"], index)

    coarse_errors = data["err"][:BASE["K"]]
    axis = fig.add_subplot(grid[1, 0])
    axis.bar(range(1, len(coarse_errors) + 1), np.minimum(coarse_errors, 30), color=["green" if e <= FOUND_SIGMA else "gray" for e in coarse_errors])
    axis.axhline(FOUND_SIGMA, color="red", ls="--")
    axis.set_xlabel("coarse rank")
    axis.set_ylabel("error (sigma, cap 30)")
    axis.set_title("the K = %d coarse peaks" % BASE["K"], fontsize=8)

    axis = fig.add_subplot(grid[1, 1])
    axis.bar(range(1, len(details["errors"]) + 1), np.minimum(details["errors"], 30), color=["green" if e <= FOUND_SIGMA else "gray" for e in details["errors"]])
    axis.axhline(FOUND_SIGMA, color="red", ls="--")
    axis.set_xlabel("new rank")
    axis.set_title("the M = %d peaks sent to the fine stage" % BASE["M"], fontsize=8)

    inliers = fine[MATCHERS[0]][index]["inliers"][details["order"]]
    axis = fig.add_subplot(grid[1, 2])
    axis.bar(range(1, len(inliers) + 1), inliers, color=["green" if e <= FOUND_SIGMA else "gray" for e in details["errors"]])
    axis.axhline(BASE["bar"], color="red", ls="--")
    axis.set_xlabel("candidate")
    axis.set_ylabel("inliers")
    axis.set_title("inliers (%s)" % MATCHERS[0], fontsize=8)

    chosen_peak = data["short"][details["order"][details["chosen"]]]
    show(fig.add_subplot(grid[1, 3]), map_piece(level_images, level, chosen_peak[1], chosen_peak[2]),
         "chosen candidate #%d, %d inliers | answer %.1f sigma off | %s" % (details["chosen"] + 1, row["inliers"], row["err_sigma"], "ANSWER" if row["answered"] else "SKIP"))
    fig.suptitle("%s | photo %d | height %.0f m | level %d" % (label, index, record["height"], level), fontsize=10)
    plt.tight_layout()
    plt.show()
    del inputs
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


table = base[MATCHERS[0]]
kinds = [
    ("target not in the K coarse peaks", table[~table["in_K"]].sort_values("err_sigma", ascending=False)),
    ("target in K but not in the M sent to fine", table[table["in_K"] & ~table["in_M"]]),
    ("target in M but the wrong candidate was chosen", table[table["in_M"] & ~table["chosen_right"]]),
    ("right candidate chosen but skipped (too few inliers)", table[table["chosen_right"] & ~table["answered"]]),
    ("answered, but wrong place", table[table["answered"] & (table["err_sigma"] > FOUND_SIGMA)]),
    ("good: answered within 1 sigma", table[table["answered"] & (table["err_sigma"] <= STRICT_SIGMA)]),
]
for label, part in kinds:
    print("=" * 100)
    print("%s: %d photos" % (label, len(part)))
    count_shown = N_SAMPLES if not label.startswith("good") else min(2, N_SAMPLES)
    for photo in list(part["photo"])[:count_shown]:
        draw_sample(int(photo), label)

# Part 10 · Save the tables

The per-photo table of the baseline (one row per photo and matcher) and the sweep tables go to `tests/Stage_Evaluation/`, so settings can be compared later without running anything again.

In [ ]:
out_dir = ROOT / "tests" / "Stage_Evaluation"
out_dir.mkdir(parents=True, exist_ok=True)
tag = "%s_%s_K%d_M%d" % (DATA, PRIMARY, K_BASE, M_BASE)

for name in MATCHERS:
    table = base[name].copy()
    table["height"] = [record_by_index[p]["height"] for p in table["photo"]]
    table["valid_frac"] = [record_by_index[p]["valid_frac"] for p in table["photo"]]
    table["contrast"] = [record_by_index[p]["contrast"] for p in table["photo"]]
    table["edge_strength"] = [record_by_index[p]["edge_strength"] for p in table["photo"]]
    table["near_z"] = [record_by_index[p]["methods"][PRIMARY]["stats"]["near_z"] for p in table["photo"]]
    table["map_max_err_sigma"] = [record_by_index[p]["methods"][PRIMARY]["stats"]["argmax_err"] for p in table["photo"]]
    table.to_csv(out_dir / ("per_photo_%s_%s.csv" % (tag, name)), index=False)
    print("saved", out_dir / ("per_photo_%s_%s.csv" % (tag, name)))

sweep_kmw.to_csv(out_dir / ("sweep_K_M_weight_%s_%s.csv" % (tag, MATCHERS[0])), index=False)
sweep_bar.to_csv(out_dir / ("sweep_bar_gamma_rescue_%s.csv" % tag), index=False)
map_table.to_csv(out_dir / ("maps_%s.csv" % tag), index=False)
print("saved the sweep tables and the map table in", out_dir)


**Caveats.** The photos and the map are from the development set; the real evaluation uses a different map. With 100 photos every share has an interval of several percentage points (for a share near 0.9 about plus or minus 0.06), and the sweeps are searched on the photos they are scored on. The learned fusion was trained on the NUDT (Changsha) photos, so the official set tests it on another city. The fine stage was run on the coarse order of the primary method; the other coarse method is evaluated up to stage 3 only. To compare data sets or seeds, run the notebook again with the setting changed and compare the saved tables photo by photo.

# Part 11 · Colab: copy the results to Drive

In [ ]:
#main
# Copies the result tables of this run from the local disk to Drive. A separate folder: nothing is written into the repo copy.
import subprocess, os
source = LOCAL_REPO + "/tests/Stage_Evaluation"
assert os.path.isdir(source), "no results yet: run the whole notebook first"
target = DRIVE_RESULTS + "/Stage_Evaluation"
os.makedirs(target, exist_ok=True)
print(subprocess.run('rsync -rt "%s/" "%s/" && ls -la "%s"' % (source, target, target), shell=True, capture_output=True, text=True, executable="/bin/bash").stdout)